![icon](../docs/assets/logo.svg)

<p align="center">
  <img src="https://img.shields.io/badge/MongoDB-8.0.4_(CP_·_Docs)-00ED64?style=for-the-badge&logo=mongodb&logoColor=black"/>
  <img src="https://img.shields.io/badge/Apache_Cassandra-5.0.2_(AP_·_Colunar)-38BDF8?style=for-the-badge&logo=apachecassandra&logoColor=black"/>
  <img src="https://img.shields.io/badge/Redis-7.4.2_(AP_·_Speed_RAM)-FF4438?style=for-the-badge&logo=redis&logoColor=white"/>
  <img src="https://img.shields.io/badge/Polars-Out--of--Core-FBBF24?style=for-the-badge&logo=polars&logoColor=black"/>
</p>

---

# Notebook 02 — Ingestão Poliglota Out-of-Core (Modo A: Carga Histórica Completa até `DAY = 711`)

**Objetivo:** Provisionar os schemas e índices idempotentes e executar o pipeline Polars *out-of-core* (`scan_csv` + `read_csv_batched`) populando:
1. **MongoDB (`households`)**: todos os **2.500 domicílios** (`801` com `demographics`, `1.699` sem `demographics`), `behavior_summary` e `recent_baskets` (últimas 5 cestas enriquecidas com `department` e `commodity_desc`).
2. **Apache Cassandra (`basketlens`)**: as 4 tabelas orientadas à consulta (`transactions_by_household`, `transactions_by_household_product`, `sales_by_product_week`, `causal_promotions_by_store_week`).
3. **Redis**: campanhas e cupons vigentes em `REFERENCE_DAY = 650` com **TTL** proporcional à vigência restante + vitrines semanais (`ranking:week:94`, `store:{id}:week:94:displays`).
4. **Auditoria de Índices (`explain("executionStats")`)**: comprovação de `IXSCAN` / `EXPRESS_IXSCAN` e zero `COLLSCAN` no MongoDB.

### 1. `check_all_connections()` e `initialize_all_schemas()` — Interação com os Bancos
```text
                        ┌───► [MongoDB :27017]   ── admin.command("ping") + 3 índices B-Tree ──► True
check_all_connections() ├───► [Cassandra :9042]  ── KEYSPACE basketlens + 4 tabelas CQL ───────► True
initialize_all_schemas()└───► [Redis :6379]      ── PING (chaves efêmeras c/ TTL sob demanda) ─► True
```

In [1]:
import pprint
from basketlens import check_all_connections, initialize_all_schemas
from basketlens.connections import cassandra_session_scope, get_mongo_client, get_mongo_db, get_redis_client
from basketlens.ingestion import run_full_ingestion
from basketlens.schemas.mongo_schema import HOUSEHOLDS_COLLECTION, explain_query

print("1) Healthcheck dos 3 motores NoSQL:")
pprint.pprint(check_all_connections())

print("\n2) Inicialização idempotente de schemas e índices:")
pprint.pprint(initialize_all_schemas())

1) Healthcheck dos 3 motores NoSQL:
{'cassandra': True, 'mongodb': True, 'redis': True}

2) Inicialização idempotente de schemas e índices:


{'cassandra_tables': ['causal_promotions_by_store_week',
                      'sales_by_product_week',
                      'transactions_by_household',
                      'transactions_by_household_product'],
 'mongo_indexes': ['_id_',
                   'idx_behavior_avg_weekly_spend_desc',
                   'idx_demographics_class1_class3',
                   'idx_household_key_unique'],
 'redis_ready': True}


## 2. Execução do Pipeline de Ingestão Poliglota (`cutoff_day = 711`, `reference_day = 650`)

```text
run_full_ingestion(cutoff_day=711, reference_day=650, seconds_per_day=5)
  ├──► 1. ingest_mongo_households() ──► [MongoDB]   Upsert em lote de 2.500 docs (801 c/ demo, $slice:-5)
  ├──► 2. ingest_cassandra_data()   ──► [Cassandra] UNLOGGED BATCH nas 4 tabelas CQL por partição
  └──► 3. ingest_redis_state()      ──► [Redis]     Pipeline HSET/SADD/ZADD + EXPIRE (TTL proporcional)
```

In [2]:
# Para execução ágil e determinística em notebook/CI, processamos integralmente os 2.500
# perfis no MongoDB (100% das 2,59M transações via Polars streaming), 100% da vigência no Redis
# e lotes representativos + 100% dos domicílios demográficos (801), John Smith (#208) e produto #72717 no Cassandra.
# Para ingerir todos os 52 lotes transacionais e 736 lotes causais sem limite de lotes, basta passar tx_max_batches=None, causal_max_batches=None.
summary = run_full_ingestion(
    cutoff_day=711,
    reference_day=650,
    seconds_per_day=5,
    include_causal=True,
    tx_max_batches=1,
    sales_max_rows=20000,
    causal_max_batches=1,
)
pprint.pprint(summary)

{'cassandra': {'causal_promotions': {'batches_processed': 1,
                                     'cutoff_week': 102,
                                     'rows_written': 58819},
               'cutoff_day': 711,
               'cutoff_week': 102,
               'sales_by_product_week': {'cutoff_day': 711,
                                         'rows_written': 20000},
               'transactions': {'batches_processed': 1,
                                'cutoff_day': 711,
                                'rows_written': 76401}},
 'cutoff_day': 711,
 'elapsed_seconds': 57.499,
 'mongodb': {'batches_written': 5,
             'cutoff_day': 711,
             'total_documents': 2500,
             'with_demographics': 801,
             'without_demographics': 1699},
 'redis': {'active_campaign_count': 5,
           'active_campaigns': {15: {'campaign': 15,
                                     'description': 'TypeC',
                                     'end_day': 708,
                     

## 3. Verificação dos Dados Carregados nos 3 Bancos (Estudo de Caso `household_key = 208`)

In [3]:
mc = get_mongo_client()
db = get_mongo_db(client=mc)
hh208_doc = db[HOUSEHOLDS_COLLECTION].find_one({"household_key": 208}, {"_id": 0})
print("Documento MongoDB (household_key = 208 — John Smith):")
pprint.pprint({
    "household_key": hh208_doc["household_key"],
    "demographics": hh208_doc["demographics"],
    "behavior_summary": hh208_doc["behavior_summary"],
    "campaigns_received": hh208_doc["campaigns_received"],
    "recent_baskets_count": len(hh208_doc["recent_baskets"]),
    "most_recent_basket": hh208_doc["recent_baskets"][-1],
})
mc.close()

with cassandra_session_scope(keyspace="basketlens") as session:
    rows = list(
        session.execute(
            "SELECT week_no, day, basket_id, product_id, quantity, sales_value "
            "FROM transactions_by_household WHERE household_key = 208 LIMIT 5"
        )
    )
    print("\nCassandra transactions_by_household (household_key = 208, top 5 recentes):")
    for r in rows:
        print(r)

r_client = get_redis_client()
print("\nRedis estado de vigência em REFERENCE_DAY = 650:")
print("Campanha 22 HASH:", r_client.hgetall("campanha:22"))
print("Campanha 22 TTL (s):", r_client.ttl("campanha:22"))
print("household:208:campaigns:", r_client.smembers("household:208:campaigns"))
print("Top 5 commodities na Semana 94 (ZSET):", r_client.zrevrange("ranking:week:94", 0, 4, withscores=True))
r_client.close()

Documento MongoDB (household_key = 208 — John Smith):
{'behavior_summary': {'active_weeks': 80,
                      'avg_basket_value': 30.69,
                      'avg_weekly_spend': 53.7,
                      'first_day': 49,
                      'last_day': 701,
                      'top_commodities': ['COUPON/MISC ITEMS',
                                          'BEEF',
                                          'ICE CREAM/MILK/SHERBTS'],
                      'top_departments': ['GROCERY', 'MEAT', 'KIOSK-GAS'],
                      'total_baskets': 140,
                      'total_spend': 4296.33,
                      'updated_at': '2026-10-19T01:20:02Z'},
 'campaigns_received': [8, 13, 17, 18, 22, 26, 29, 30],
 'demographics': {'age_group': 'Age Group4',
                  'classification_1': 'Age Group4',
                  'classification_2': 'Y',
                  'classification_3': 'Level5',
                  'classification_4': '2',
                  'classification_

## 4. Auditoria de Planos de Execução no MongoDB (`explain("executionStats")`)

Comprova que as 3 consultas críticas utilizam `IXSCAN` / `EXPRESS_IXSCAN` e examinam apenas as chaves necessárias (**zero `COLLSCAN`**).

```text
explain_query(collection, filter_spec, sort_spec)
  └──► [MongoDB db.households] ──► .find(filter).sort(sort).explain("executionStats")
                               ◄── stage = "EXPRESS_IXSCAN" | "IXSCAN" (0 ms, zero COLLSCAN)
```

In [4]:
mc = get_mongo_client()
db = get_mongo_db(client=mc)
coll = db[HOUSEHOLDS_COLLECTION]

plan_hh = explain_query(coll, {"household_key": 208})
plan_demo = explain_query(
    coll,
    {
        "demographics.classification_1": "Age Group2",
        "demographics.classification_3": "Level6",
    },
)
plan_spend = explain_query(
    coll,
    {"behavior_summary.avg_weekly_spend": {"$gte": 50.0}},
    sort_spec=[("behavior_summary.avg_weekly_spend", -1)],
)

for label, plan in [
    ("1. Busca por household_key = 208", plan_hh),
    ("2. Filtro composto demográfico (classification_1 + classification_3)", plan_demo),
    ("3. Ranking por avg_weekly_spend DESC", plan_spend),
]:
    winning = plan["queryPlanner"]["winningPlan"]
    stats = plan["executionStats"]
    print(f"=== {label} ===")
    print("Winning Plan:", winning)
    print(
        f"nReturned={stats['nReturned']} | "
        f"totalKeysExamined={stats['totalKeysExamined']} | "
        f"totalDocsExamined={stats['totalDocsExamined']} | "
        f"executionTimeMillis={stats['executionTimeMillis']}\n"
    )

mc.close()

=== 1. Busca por household_key = 208 ===
Winning Plan: {'isCached': False, 'stage': 'EXPRESS_IXSCAN', 'keyPattern': '{ household_key: 1 }', 'indexName': 'idx_household_key_unique'}
nReturned=1 | totalKeysExamined=1 | totalDocsExamined=1 | executionTimeMillis=0

=== 2. Filtro composto demográfico (classification_1 + classification_3) ===
Winning Plan: {'isCached': False, 'stage': 'FETCH', 'inputStage': {'stage': 'IXSCAN', 'keyPattern': {'demographics.classification_1': 1, 'demographics.classification_3': 1}, 'indexName': 'idx_demographics_class1_class3', 'isMultiKey': False, 'multiKeyPaths': {'demographics.classification_1': [], 'demographics.classification_3': []}, 'isUnique': False, 'isSparse': False, 'isPartial': False, 'indexVersion': 2, 'direction': 'forward', 'indexBounds': {'demographics.classification_1': ['["Age Group2", "Age Group2"]'], 'demographics.classification_3': ['["Level6", "Level6"]']}}}
nReturned=13 | totalKeysExamined=13 | totalDocsExamined=13 | executionTimeMillis=